<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 1 精华 — Scoping(澄清 + Brief 生成)

**一句话定位**:**研究开始前先把意图问清楚** —— 用 `with_structured_output` 决定要不要追问,用 `Command(goto, update)` 优雅地路由,最后产出一份详细的 research brief 给后续阶段。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 为啥要专门搞一个「scoping」阶段?**

做研究前,用户给的请求通常 **信息不全**:

| 用户说 | 实际缺什么 |
|--------|----------|
| 「**研究一下 AI safety**」 | 受众是技术读者还是高管?要多深?要哪些公司? |
| 「**对比这几家公司**」 | 对比哪些维度?时间窗口?引用要求? |
| 「**写个产品报告**」 | 给谁看?多长?要数据图表吗? |

**不澄清 → 研究跑偏 → 浪费 5 分钟 + 一堆 token**。Scoping 就是在 **投入研究时间前**,把这些问题问清楚。

本节的两步流程:

1. **User Clarification** — LLM 判断是否需要追问
2. **Brief Generation** — 把对话转成 **详细的 research brief**

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🏗️ Scoping workflow 的架构

</div>

**整体数据流**:

```
User 输入(messages)
        │
        ▼
┌──────────────────────────────────────┐
│  clarify_with_user                   │
│  ─ 调 LLM(with_structured_output)   │
│  ─ 输出 ClarifyWithUser:             │
│    { need_clarification: bool,       │
│      question: str / verification }  │
└──┬───────────────────────────────────┘
   │
   ├─ need_clarification=True ──▶ END(向用户追问,等下一轮)
   │
   └─ need_clarification=False ──▶ write_research_brief
                                        │
                                        ▼
                                 ┌──────────────────┐
                                 │ 调 LLM 写 brief  │
                                 │ → 写入 state     │
                                 └──┬───────────────┘
                                    │
                                    ▼
                                   END(brief 准备好,可以开始研究)
```

| 节点 | 作用 | 用什么技术 |
|------|------|----------|
| `clarify_with_user` | 决定要不要追问 | `with_structured_output(ClarifyWithUser)` + `Command` |
| `write_research_brief` | 把对话转成 brief | LLM + 写 state |
| **State** | `AgentState` 含 messages + research_brief + ... | 继承 `MessagesState` |
| **Input schema** | `AgentInputState`(只暴露 messages) | 接口隔离 |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔑 技术核心 ① — `with_structured_output` + Pydantic schema

</div>

**问题**:怎么让 LLM 做「**要不要追问**」这种二选一决策?

**答案**:**让它输出结构化对象**,而不是自然语言。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from pydantic import BaseModel, Field

class ClarifyWithUser(BaseModel):
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">need_clarification: bool</span> = Field(description="是否需要追问用户")
    question: str = Field(description="如果需要追问,问什么(否则为 '')")
    verification: str = Field(description="如果不需要追问,确认理解了什么")

# 把 schema 绑定给 LLM
structured_llm = model.<span style="background:#3d3414; color:#fde047; padding:0 2px;">with_structured_output(ClarifyWithUser)</span>

result = structured_llm.invoke([...])
# result 是 ClarifyWithUser 实例,可以直接 .need_clarification 取值
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 Field 的 `description` 不是装饰,是 LLM 看的「**字段说明书**」**

`Field(description="...")` 里的描述 **会作为 JSON schema 的一部分传给 LLM**。写得清楚,LLM 输出就准;写得模糊,LLM 就会乱填。

**实战建议**:

- ✅ 写 **「**何时该这个值**」**:`description="是否需要追问用户。如果用户请求清晰具体,设 False;如果有歧义或信息不全,设 True"`
- ❌ 不要写 **「**这是个布尔值**」**(废话,LLM 看 type hint 就知道)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔀 技术核心 ② — `Command` 一行同时更 state + 跳转

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langgraph.types import Command
from typing_extensions import Literal

def clarify_with_user(state: AgentState) -> <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command[Literal["write_research_brief", "__end__"]]</span>:
    result = structured_llm.invoke([...])
    
    if result.need_clarification:
        return Command(
            <span style="background:#3d3414; color:#fde047; padding:0 2px;">goto=END</span>,
            update={"messages": [AIMessage(content=result.question)]}
        )
    else:
        return Command(
            <span style="background:#3d3414; color:#fde047; padding:0 2px;">goto="write_research_brief"</span>,
            update={"messages": [AIMessage(content=result.verification)]}
        )
</code></pre>

**`Command` 同时做 2 件事**:

| 字段 | 作用 |
|------|------|
| `goto` | 下一个 node(或 `END`) |
| `update` | state 更新 dict |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 为啥比传统 `conditional_edges` 优雅?**

传统写法需要 **3 步**:

1. 节点写 state(把 decision 存进去)
2. `conditional_edges` 函数 **再读 state** 决定 goto
3. 在图里 `.add_conditional_edges(...)`

`Command` 写法 **1 步搞定**:节点直接 `return Command(goto=..., update=...)`。

→ 「**算决策的地方**」和「**决策落地的地方**」**绑在一起**,代码更紧凑、更可读。

📎 详细对比:[`0_b_⭐️_edges_vs_Command`(隔壁项目)](../../project-002-ambiant-agents/notebooks/0_b_⭐️_edges_vs_Command.ipynb)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧩 技术核心 ③ — `AgentInputState(MessagesState): pass` 接口隔离

</div>

本课最容易让人 **看一眼跳过** 的代码:

```python
class AgentInputState(MessagesState):
    """Input state for the full agent."""
    pass
```

看起来什么都没干?**实际上做了一件超重要的事 —— 接口隔离**。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 关键一行 ↓
deep_researcher_builder = StateGraph(
    AgentState,                                  # 内部完整 state(很多字段)
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">input_schema=AgentInputState</span>          # 对外只暴露 messages
)
</code></pre>

**两层 schema 各自的职责**:

| Schema | 谁用 | 包含啥 |
|--------|------|--------|
| `AgentInputState` | **外部调用方**(用户、API) | **只有 `messages`** |
| `AgentState` | **节点内部** | messages + `research_brief` + `raw_notes` + `final_report` + ... |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 为什么是 `pass`?**

因为继承 `MessagesState` 已经引入 `messages` 字段了。`pass` 的意思是「**我什么都不加,但我要一个独立的类型身份**」。

它是个 **类型层面的别名**,作用 = **API 边界文档**:

- ✅ **明确边界** — 一眼看清「**这个 graph 只接收 messages**」
- ✅ **future-proof** — 想加输入字段?只改 `AgentInputState`,不污染内部 state
- ✅ **类型安全** — IDE 提示更准

**类比**:像 Pydantic 的 `class CreateUserRequest(BaseModel): pass` —— 即便现在跟内部 User 模型完全一样,**接口要有独立身份**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📝 Research Brief 的「好坏」标准

</div>

Brief 的质量 **直接决定后续研究的方向**。本节定义了两条评估标准:

| ✅ 必须做 | ❌ 不能做 |
|---------|---------|
| **抓住对话里所有 relevant criteria** | **臆造、假设用户没说过的 criteria** |
| 时间窗口、来源限制、受众、深度... | 「我猜用户想要...」 |
| 用具体的语言而不是模糊概括 | 「全面研究一下 X」(没意义) |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 第二条「不臆造」**比第一条更难做到

LLM 倾向 **「**帮你补充**」**信息。比如用户只说「研究 AI safety」,LLM 可能把 brief 写成「**Compare AI safety approaches at OpenAI, Anthropic, and DeepMind**」—— 这 **3 家公司是 LLM 加的,不是用户说的**。

后果:研究方向被 LLM 替用户决定了,**用户可能根本不想看这 3 家**。

**解决办法**:

1. **prompt 里强调** 「**只用用户明确提到的 criteria**」
2. **用 LLM-as-judge 评估** brief 有没有引入新假设(本节示范)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 LLM-as-Judge — 评估 Brief 质量

</div>

本节用 LangSmith + LLM-as-judge 评估 brief。**关键技巧来自 [Hamel Husain 的指南](https://hamel.dev/blog/posts/llm-judge/index.html)**:

### 🎯 LLM-as-Judge 的 7 个技巧

| # | 技巧 | 一句话 |
|---|------|--------|
| 1 | **专家角色** | 「research brief evaluator」,**别用泛泛的 "helpful assistant"** |
| 2 | **二元 pass/fail** | 避免「1-10 分」之类复杂评分(LLM 主观波动大) |
| 3 | **丰富 context** | 解释 brief 质量为何重要 |
| 4 | **XML 结构** | `<role>` / `<task>` / `<inputs>` / `<guidelines>` |
| 5 | **正反例** | 至少 3-4 个例子,**正反都有** |
| 6 | **明确输出格式** | 必须用 structured output |
| 7 | **「**拿不准时倾向 FAIL**」** | 保守评估,减少假阳性 |

### 📊 实验流程

| 步骤 | 操作 |
|------|------|
| **1** | 创建 LangSmith dataset(2 个对话示例 + 真值 criteria) |
| **2** | 写 2 个 evaluator:**抓住相关 criteria** + **不臆造新假设** |
| **3** | 跑实验,看分数 |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 为啥要做 eval?**

- ✅ **验证每一步符合预期** — 不只看最终结果,**单步也要测**
- ✅ **了解 latency 和成本** — LangSmith 自动统计
- ✅ **试更便宜的模型** — 测试小模型能否完成 scoping 任务(常常可以)
- ✅ **调 prompt 时有回归测试** — 改完一跑就知道有没有变差

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比 — Scoping 像什么?

</div>

### 🅰️ 类比:咨询顾问的「**Intake Interview**」

好的咨询顾问 **不会马上动手做项目**。他们先问:

- 「**你想解决什么具体问题?**」
- 「**给谁看?**」
- 「**时间窗口?**」
- 「**有哪些约束?**」

**然后总结成一份「**项目章程**」给客户确认**,客户点头 → 才开干。

→ 这就是 `write_research_brief` 在做的事。

### 🅱️ 类比:Google Maps 输入目的地

你输入「**北京饭店**」,Google Maps 弹出对话框:

- 「**你是说东城区的北京饭店还是大兴的北京饭店?**」
- 「**你要走车还是地铁?**」

这就是 `clarify_with_user` 在做的事 —— **歧义触发追问,清晰直接开导航**。

### 🆚 对比:**没有 scoping 阶段**

| 没 scoping | 有 scoping |
|-----------|-----------|
| 用户说「研究 X」→ agent 立刻动手 | 先问清楚才动手 |
| 研究跑偏不知道,**直到看到结果才发现** | **风险前置**,不浪费下游 token |
| 用户接着说「不,我是想要 Y」 → **从头来一遍** | 一次准 |
| 多轮交互体验差 | **像跟真人助理对话** |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 4 个最容易踩的坑**

1. **❌ Pydantic Field 写 description 时过于简略** → LLM 输出 schema 对但语义不对(如「确认理解」字段反而填了追问问题)。**description 要写「**何时该这个值**」**

2. **❌ 直接用 `StateGraph(AgentState)` 不指定 `input_schema`** → 外部调用方理论上能传 internal 字段(`research_brief` 等),API 边界混乱

3. **❌ brief 生成时不强调「**只用用户提到的**」**→ LLM **替用户加假设** → 研究方向跑偏

4. **❌ LLM-as-judge 不用 structured output** → 解析裸文本失败,评估系统脆弱。**永远用 `with_structured_output`**

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Scoping 的本质 = 「**前置澄清,延后投入**」**:

- **用 `with_structured_output`** 把「要不要追问」变成 LLM 输出的字段(`bool`)
- **用 `Command(goto, update)`** 把「决策 + 路由 + state 写入」一行搞定
- **用 `AgentInputState(MessagesState): pass`** 给 graph 做接口隔离(API 边界 vs 内部 state)
- **用 LangSmith + LLM-as-judge** 检验 brief 质量(抓住 criteria + 不臆造)

### 🎯 学完这节,你应该能回答:

1. ✅ **为啥 Field 的 description 这么重要?**(LLM 看 schema 决定字段语义)
2. ✅ **`Command` 比 conditional_edges 优雅在哪?**(算决策 + 路由绑一起)
3. ✅ **`AgentInputState: pass` 不是废代码,作用是什么?**(API 边界 + 类型身份)
4. ✅ **怎么防止 LLM 给 brief 臆造?**(prompt 强调 + LLM-as-judge 验证)
5. ✅ **LLM-as-judge 必须用 structured output 吗?**(必须,否则解析脆弱)

📎 配套阅读:[`1_scoping.ipynb` 主课](./1_scoping.ipynb) · 下一节 [`2_z_⭐️_本节精华_ResearchAgent.ipynb`](./2_z_⭐️_本节精华_ResearchAgent.ipynb)

</div>